# Deployment LLM to HuggingFace

## Import library

In [1]:
import os
import json
import shutil
import tempfile
import textwrap

from huggingface_hub import HfApi, create_repo, upload_folder, upload_file, whoami, ModelCard, DatasetCard
from dotenv import load_dotenv
from pathlib import Path

In [2]:
# Load configuration
BASE   = Path.cwd().parents[2]
PROJ   = BASE / "credit_risk_production"
DS     = BASE / "data_science"

load_dotenv(PROJ / ".env")

HF_TOKEN = os.getenv("HUGGINGFACE_API_KEY")
HF_USER = whoami(token=HF_TOKEN)["name"]

MODEL_REPO = f"{HF_USER}/credit-risk-challengers"
DATA_REPO  = f"{HF_USER}/credit-risk-merged-data"
RAG_REPO   = f"{HF_USER}/credit-risk-policy-rag"
AUDIT_REPO = f"{HF_USER}/credit-risk-audit"

# Local artifact roots
MODEL_ROOT = PROJ / "models" / "credit_risk"
DATA_FILE  = PROJ / "database" / "data" / "merged_credit_risk_data.parquet"
PDF_DIR    = PROJ / "database" / "pdf"
AUDIT_DIR  = DS / "reports" / "audit"

api = HfApi(token=HF_TOKEN)
print(f"user={HF_USER}")
for r in (MODEL_REPO, DATA_REPO, RAG_REPO, AUDIT_REPO):
    print(" ", r)

user=Mipjeiger
  Mipjeiger/credit-risk-challengers
  Mipjeiger/credit-risk-merged-data
  Mipjeiger/credit-risk-policy-rag
  Mipjeiger/credit-risk-audit


## Verify the condition

In [3]:
# 1. Preconditions — fail fast before creating remote repos
checks = {
    "model_root":  MODEL_ROOT.exists(),
    "data_file":   DATA_FILE.exists(),
    "pdf_dir":     PDF_DIR.exists(),
    "audit_dir":   AUDIT_DIR.exists(),
    "token_ok":    bool(HF_TOKEN),
}
for k, v in checks.items():
    print(f"{'✅' if v else '❌'} {k} = {v}")

if not all(checks.values()):
    raise RuntimeError("Missing prerequisites — fix before continuing.")

# Sanity: what will we actually upload?
def dir_size_mb(p: Path) -> float:
    return round(sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / 1e6, 2)

print("\nsizes (MB):")
print(f"  models : {dir_size_mb(MODEL_ROOT)}")
print(f"  data   : {round(DATA_FILE.stat().st_size/1e6, 2)}")
print(f"  pdfs   : {dir_size_mb(PDF_DIR)}")
print(f"  audit  : {dir_size_mb(AUDIT_DIR)}")

✅ model_root = True
✅ data_file = True
✅ pdf_dir = True
✅ audit_dir = True
✅ token_ok = True

sizes (MB):
  models : 199.16
  data   : 2.91
  pdfs   : 3.54
  audit  : 0.01


## Build the staging tree for the model repo

In [4]:
STAGE = Path(tempfile.mkdtemp(prefix="hf_models_"))
STAGE.mkdir(exist_ok=True, parents=True)

# Copy models + metadata + metrics
shutil.copytree(MODEL_ROOT / "ml_credit_risk", STAGE / "models")
shutil.copytree(MODEL_ROOT / "metadata_credit_risk", STAGE / "metadata")
shutil.copytree(MODEL_ROOT / "metrics_credit_risk", STAGE / "metrics")
shutil.copytree(MODEL_ROOT / "params_credit_risk", STAGE / "params")
shutil.copy(MODEL_ROOT / "confusion_matrices.npz", STAGE / "confusion_matrices.npz")

# requirements loaders
(STAGE / "requirements.txt").write_text("\n".join([
    "scikit-learn>=1.3",
    "xgboost>=2.0",
    "pandas>=2.0",
    "numpy>=1.26",
    "joblib>=1.3",
]))

# 2c. loading example
(STAGE / "load_example.py").write_text(textwrap.dedent('''
    """Minimal loader for credit-risk challenger models."""
    import json, joblib
    from pathlib import Path
    from huggingface_hub import snapshot_download

    repo = "REPO_ID"
    local = snapshot_download(repo_id=repo)
    root = Path(local)

    meta = json.loads((root / "metadata" / "metadata.json").read_text())
    models = {name: joblib.load(root / "models" / fname)
              for name, fname in meta["model_files"].items()}

    # Example: score a dataframe X with the champion
    champion = meta["model_files"]  # pick whichever you want
    print("loaded:", list(models))
'''.replace("REPO_ID", MODEL_REPO)))

print("staged files:")
for p in sorted(STAGE.rglob("*")):
    if p.is_file():
        print(" ", p.relative_to(STAGE))

staged files:
  confusion_matrices.npz
  load_example.py
  metadata/metadata.json
  metrics/model_metrics.csv
  models/decision_tree.joblib
  models/gradient_boosting.joblib
  models/k_nearest_neighbors.joblib
  models/logistic_regression.joblib
  models/random_forest.joblib
  models/xgboost.joblib
  params/best_parameters.json
  requirements.txt


## Write model card

In [38]:
# 3. Model card
card = ModelCard(f"""
---
library_name: scikit-learn
tags:
  - credit-risk
  - tabular-classification
  - xgboost
  - scikit-learn
  - finance
---

# Credit Risk Challenger Models

Six benchmark classifiers trained on the merged credit-risk dataset,
used as **challengers** against the production "champion" scorecard.

## Models
| Name | File | Notes |
|---|---|---|
| Logistic Regression | `logistic_regression.joblib` | linear baseline |
| Random Forest       | `random_forest.joblib`       | bagged trees |
| Gradient Boosting   | `gradient_boosting.joblib`   | sklearn GBM |
| XGBoost             | `xgboost.joblib`             | primary challenger |
| K-Nearest Neighbors | `k_nearest_neighbors.joblib` | distance-based |
| Decision Tree       | `decision_tree.joblib`       | interpretable baseline |

## Target
`Approved_Flag` — 4-class ordinal (classes: {json.dumps(CLASS_LABELS) if 'CLASS_LABELS' in dir() else '[0,1,2,3]'}).

## Features
{len(json.loads((MODEL_ROOT/'metadata_credit_risk'/'metadata.json').read_text())['feature_columns'])} features. See `metadata/metadata.json`.

## Metrics
See `metrics/model_metrics.csv` (accuracy, precision, recall, F1, ROC AUC).

## Usage
""")

{ (STAGE/'load_example.py').read_text() }

{'\n"""Minimal loader for credit-risk challenger models."""\nimport json, joblib\nfrom pathlib import Path\nfrom huggingface_hub import snapshot_download\n\nrepo = "Mipjeiger/credit-risk-challengers"\nlocal = snapshot_download(repo_id=repo)\nroot = Path(local)\n\nmeta = json.loads((root / "metadata" / "metadata.json").read_text())\nmodels = {name: joblib.load(root / "models" / fname)\n          for name, fname in meta["model_files"].items()}\n\n# Example: score a dataframe X with the champion\nchampion = meta["model_files"]  # pick whichever you want\nprint("loaded:", list(models))\n'}

## Push model repo

In [43]:
from datetime import datetime

# 4. Push model repo
create_repo(repo_id=MODEL_REPO, repo_type="model", exist_ok=True, token=HF_TOKEN)

upload_folder(
    repo_id=MODEL_REPO,
    folder_path=str(STAGE),
    repo_type="model",
    token=HF_TOKEN,
    commit_message=f"challengers upload {datetime.now():%Y-%m-%d %H:%M}",
    ignore_patterns=["*.tmp", "__pycache__/*", ".ipynb_checkpoints/*"],
)
print(f"✅ https://huggingface.co/{MODEL_REPO}")

✅ https://huggingface.co/Mipjeiger/credit-risk-challengers


## Push the dataset repo

In [62]:
import pandas as pd
from datasets import load_dataset
from datetime import timezone
from sqlalchemy import create_engine, text

def push_dataset_bundle(stage_dir: Path, repo_id: str, card_content: str, commit_tag: str):
    """Save dataset card, create HF repo, and upload staged directory."""
    card = DatasetCard(card_content)
    (stage_dir / "README.md").write_text(str(card))

    # Create & push to HF repo
    create_repo(repo_id=repo_id, repo_type="dataset", exist_ok=True, token=HF_TOKEN)
    upload_folder(
        repo_id=repo_id,
        folder_path=str(stage_dir),
        repo_type="dataset",
        token=HF_TOKEN,
        commit_message=f"{commit_tag} upload {datetime.now(timezone.utc):%Y-%m-%d %H:%M}"
    )
    print(f"✅ Uploaded to: https://huggingface.co/datasets/{repo_id}")


# ==============================================================================
# 1. Stage & Push Tabular Dataset (DATA_REPO)
# ==============================================================================
SAMPLE_ROWS = None
DS_STAGE = Path(tempfile.mkdtemp(prefix="hf_data_"))

df_full = pd.read_parquet(DATA_FILE)
df_out = df_full.head(SAMPLE_ROWS) if SAMPLE_ROWS else df_full

DATA_OUT = DS_STAGE / "data"
DATA_OUT.mkdir(parents=True, exist_ok=True)
df_out.to_parquet(DATA_OUT / "merged_credit_risk_data.parquet", index=False)

# Metadata & Stats
schema = {c: str(t) for c, t in df_full.dtypes.items()}
(DS_STAGE / "schema.json").write_text(json.dumps(schema, indent=2))

stats = {
    "rows_full": int(len(df_full)),
    "rows_shipped": int(len(df_out)),
    "columns": int(df_full.shape[1]),
    "target_distribution": {str(k): int(v) for k, v in df_full["Approved_Flag"].value_counts().items()},
    "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds")
}
(DS_STAGE / "stats.json").write_text(json.dumps(stats, indent=2))

# Card Content
ds_card_str = f"""---
language: [en]
tags: [credit-risk, tabular, finance]
---
# Credit Risk — Merged Dataset
Rows shipped: **{len(df_out):,}** of {len(df_full):,}.
Columns: **{df_full.shape[1]}**.

## Files
- `data/merged_credit_risk_data.parquet`
- `schema.json` — column → dtype
- `stats.json` — counts and target distribution

## Target
`Approved_Flag` — {json.dumps(stats["target_distribution"])}

## Usage
```python
from datasets import load_dataset
ds = load_dataset("{DATA_REPO}", split="train")
"""

# Push tabular dataset bundle to HF
push_dataset_bundle(stage_dir=DS_STAGE, repo_id=DATA_REPO, card_content=ds_card_str, commit_tag="tabular dataset")

✅ Uploaded to: https://huggingface.co/datasets/Mipjeiger/credit-risk-merged-data


## Stage & Push Audit Artifacts Bundle (AUDIT_REPO)

In [65]:
AUDIT_STAGE = Path(tempfile.mkdtemp(prefix="hf_audit_"))
AUDIT_STAGE.mkdir(parents=True, exist_ok=True)

# Copy audit artifacts and markdown reports
for f in AUDIT_DIR.glob("*.json"):
    shutil.copy(f, AUDIT_STAGE / f.name)

for f in (DS / "reports").glob("credit_risk_report_*.md"):
    shutil.copy(f, AUDIT_STAGE / f.name)

audit_manifest = {
"generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
"files": sorted(p.name for p in AUDIT_STAGE.iterdir())
}
(AUDIT_STAGE / "manifest.json").write_text(json.dumps(audit_manifest, indent=2))

audit_card_str = f"""---
tags: [audit, credit-risk, model-evaluation, reproducibility]
Credit Risk — Audit Bundle
Snapshot of model + LLM audit artifacts.

audit_*.json — A/B test, SHAP/LIME, LLM faithfulness, determinism

credit_risk_report_*.md — generated Board report

manifest.json — inventory

Generated at {audit_manifest["generated_at"]}.
"""

# Push Audit Bundle
push_dataset_bundle(stage_dir=AUDIT_STAGE, repo_id=AUDIT_REPO, card_content=audit_card_str, commit_tag="audit artifacts")

Repo card metadata block was not found. Setting CardData to empty.


✅ Uploaded to: https://huggingface.co/datasets/Mipjeiger/credit-risk-audit


In [49]:
# Build PostgreSQL Connection URL
# Load PostgreSQL configuration
from dotenv import load_dotenv
ENV_PATH = PROJ / ".env"
load_dotenv(ENV_PATH)

# PostgreSQL connection
POSTGRES_USER = os.getenv("POSTGRES_USER")
POSTGRES_PASSWORD = os.getenv("POSTGRES_PASSWORD")
POSTGRES_HOST = "localhost"
POSTGRES_PORT = os.getenv("POSTGRES_PORT")
POSTGRES_DB = os.getenv("POSTGRES_DB")
PG_URL = f"postgresql+psycopg2://{POSTGRES_USER}:{POSTGRES_PASSWORD}@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}"

## Export & Push pgvector RAG Index

In [69]:
# ==============================================================================
# 2. Export & Push pgvector RAG Index (RAG_REPO)
# ==============================================================================
RAG_STAGE = Path(tempfile.mkdtemp(prefix="hf_rag_"))

eng = create_engine(PG_URL)
with eng.connect() as conn:
    rows = conn.execute(text("""
        SELECT chunk_id, doc, page, text, embedding::text AS embedding
        FROM credit_risk.policy_chunks
    """)).mappings().all()

# Into dataframe
rag_df = pd.DataFrame(rows)

# Parse string representation "[0.1, 0.2, ...]" to list[float]
rag_df["embedding"] = rag_df["embedding"].apply(
    lambda s: [float(x) for x in s.strip("[]").split(",")] if isinstance(s, str) else s
)

# Export Parquet & JSONL formats
rag_df.to_parquet(RAG_STAGE / "chunks.parquet", index=False)
rag_df.drop(columns=["embedding"]).to_json(
    RAG_STAGE / "chunks.jsonl", orient="records", lines=True
)

# Manifest
manifest = {
    "embedder": "BAAI/bge-m3",
    "dim": int(len(rag_df["embedding"].iloc[0])),
    "sources": sorted({p.name for p in PDF_DIR.glob("*.pdf")}) if "PDF_DIR" in globals() else [],
    "n_chunks": int(len(rag_df)),
    "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds")
}
(RAG_STAGE / "manifest.json").write_text(json.dumps(manifest, indent=2))

# --- Dataset Card ---
rag_card_str = f"""---
tags:
- rag
- credit-risk
- policy
- pgvector
---
# Credit Risk — Policy RAG Index

**Embedder:** `{manifest["embedder"]}` (dim {manifest["dim"]})
**Chunks:** {manifest["n_chunks"]}
**Sources:** {", ".join(manifest["sources"])}

## Files
- `chunks.parquet` — full index including embeddings
- `chunks.jsonl`   — text + metadata, no embeddings
- `manifest.json`  — embedder, dim, sources

## Usage
```python
import pandas as pd
df = pd.read_parquet("[https://huggingface.co/datasets/](https://huggingface.co/datasets/){RAG_REPO}/resolve/main/chunks.parquet")
"""

# Push Bundle
push_dataset_bundle(
    stage_dir=RAG_STAGE,
    repo_id=RAG_REPO,
    card_content=rag_card_str,
    commit_tag="rag index"
)

# Read the RAG index back from HF
df = pd.read_parquet(f"https://huggingface.co/datasets/{RAG_REPO}/resolve/main/chunks.parquet")
print(f"✅ Successfully verified! Loaded {len(df)} rows from HF Hub.")

✅ Uploaded to: https://huggingface.co/datasets/Mipjeiger/credit-risk-policy-rag
✅ Successfully verified! Loaded 1125 rows from HF Hub.


## Deploy LLM models to HuggingFace

In [74]:
from huggingface_hub import login

login(token=HF_TOKEN)

# Deploy
upload_folder(
    folder_path=".",
    repo_id="Mipjeiger/credit-risk-challengers",
    repo_type="model",
    ignore_patterns=[                             # <- what gets skipped
        ".env", ".env.*", "*.env",
        ".ipynb_checkpoints/**", "__pycache__/**",
        ".git/**", "*.pyc", "*.tmp",
        "*.parquet",                              # don't ship raw data in a *model* repo
    ],
    commit_message="upload challenger models",
)


CommitInfo(commit_url='https://huggingface.co/Mipjeiger/credit-risk-challengers/commit/cf4c5599bab595aff9eebdd9272277cb4132a698', commit_message='upload challenger models', commit_description='', oid='cf4c5599bab595aff9eebdd9272277cb4132a698', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Mipjeiger/credit-risk-challengers', endpoint='https://huggingface.co', repo_type='model', repo_id='Mipjeiger/credit-risk-challengers'), pr_revision=None, pr_num=None)

In [75]:
!hf upload Mipjeiger/credit-risk-challengers . --exclude ".env" --exclude "*.parquet" --exclude ".ipynb_checkpoints/**"

A new version of huggingface_hub (1.32.0) is available! You are using version 1.3.5.
To update, run: curl -LsSf https://hf.co/cli/install.sh | bash -

Start hashing 1 files.
Finished hashing 1 files.
https://huggingface.co/Mipjeiger/credit-risk-challengers/commit/5356ac510f20919b7794598aa6fd03cd38fb51c7


In [76]:
from huggingface_hub import list_repo_files

files = list_repo_files("Mipjeiger/credit-risk-challengers", repo_type="model")
print("\n".join(sorted(files)))

.gitattributes
confusion_matrices.npz
huggingface_deployment.ipynb
load_example.py
metadata/metadata.json
metrics/model_metrics.csv
models/decision_tree.joblib
models/gradient_boosting.joblib
models/k_nearest_neighbors.joblib
models/logistic_regression.joblib
models/random_forest.joblib
models/xgboost.joblib
params/best_parameters.json
requirements.txt


## Deploy Scaler & Labelencoder to the HF

In [20]:
import joblib

REPO_ID = "Mipjeiger/credit-risk-challengers"

bundle = joblib.load("../../../credit_risk_production/database/LLM/outputs_llm/model_artifacts_mlflow/model_bundle.joblib")

# Write the two pieces needed for serving
joblib.dump(bundle["scaler"], "scaler.joblib")
joblib.dump(bundle["label_encoders"], "label_encoders.joblib")

# Deploy the scaler and label encoder to the same repo on hf
upload_file(
    path_or_fileobj="scaler.joblib",
    path_in_repo="metadata/scaler.joblib",
    repo_id=REPO_ID, repo_type="model", token=HF_TOKEN,
)
upload_file(
    path_or_fileobj="label_encoders.joblib",
    path_in_repo="metadata/label_encoders.joblib",
    repo_id=REPO_ID, repo_type="model", token=HF_TOKEN,
)
print("✅ scaler + encoders uploaded")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✅ scaler + encoders uploaded
